# MLflow Example: Tracking Two Model Training Runs

Dependencies are installed in the project's `.venv` (`mlflow`, `scikit-learn`, `pandas`, `jupyter`, `dagshub`). Select that venv as the notebook kernel.

## 1. Baseline: training without MLflow

In [27]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_digits

In [28]:
data = load_digits()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

def train_model(n_estimators, max_depth):
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)
    accuracy = accuracy_score(y_test, model.predict(X_test))
    print(f"Model with n_estimators={n_estimators}, max_depth={max_depth} achieved accuracy={accuracy:.4f}")
    return model, accuracy

In [29]:
print("Training Model 1...")
model_1, acc_1 = train_model(n_estimators=20, max_depth=5)

print("\nTraining Model 2...")
model_2, acc_2 = train_model(n_estimators=100, max_depth=10)

Training Model 1...
Model with n_estimators=20, max_depth=5 achieved accuracy=0.9389

Training Model 2...
Model with n_estimators=100, max_depth=10 achieved accuracy=0.9722


## 2. Now adding MLflow

### Tracking URI
**Local (steps 2–3):** start the UI first from this folder with `mlflow ui`, then **comment out the DagsHub cell** below; the notebook logs to `http://127.0.0.1:5000`.

**Remote (step 4, current setup):** run both cells. The local cell does the imports, then the DagsHub cell switches the tracking URI to DagsHub.

In [30]:
import os
import mlflow
import mlflow.sklearn
from mlflow import MlflowClient

# Local tracking server (started with `mlflow ui` from this folder)
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000"))
print("Tracking URI:", mlflow.get_tracking_uri())

Tracking URI: https://dagshub.com/Phaztox/dvc-lab_efrei.mlflow


In [31]:
# --- Step 4: remote tracking on DagsHub ---
# Code from the DagsHub repo: Remote > Experiments > "Using MLflow tracking".
# dagshub.init() only sets the MLFLOW_TRACKING_URI env variable, but the cell above already
# called mlflow.set_tracking_uri() explicitly (which wins over the env variable), so we apply it.
import dagshub
dagshub.init(repo_owner='Phaztox', repo_name='dvc-lab_efrei', mlflow=True)
mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
print(mlflow.get_tracking_uri())

Initialized MLflow to track repo "Phaztox/dvc-lab_efrei"

Repository Phaztox/dvc-lab_efrei initialized!

https://dagshub.com/Phaztox/dvc-lab_efrei.mlflow


### Create the experiment

In [32]:
# Created if it doesn't exist, and set as the active experiment
experiment_name = "RandomForest_V1.0.1"
mlflow.set_experiment(experiment_name)

<Experiment: artifact_location='mlflow-artifacts:/28c0b703260542e3b69616626e6b6923', creation_time=1795018132098, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1795018132098, lifecycle_stage='active', name='RandomForest_V1.0.1', tags={}, trace_location=None, workspace='default'>

### Log parameters, metrics and the model

In [33]:
MODEL_NAME = "random_forest_model"  # name of the model logged in each run

def train_and_log_model(n_estimators, max_depth, run_name):
    with mlflow.start_run(run_name=run_name) as run:
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)
        accuracy = accuracy_score(y_test, model.predict(X_test))

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_metric("accuracy", accuracy)
        # input_example: lets MLflow infer the model signature (needed for clean serving later)
        # skops_trusted_types: MLflow 3.18 saves sklearn models with skops (safer than pickle) and
        # refuses unknown types. We trained this model ourselves, so we trust the tree structure.
        model_info = mlflow.sklearn.log_model(
            model,
            name=MODEL_NAME,
            input_example=X_train[:5],
            skops_trusted_types=["sklearn.tree._tree.Tree"],
        )

        print(f"Logged run '{run_name}' (id={run.info.run_id}) "
              f"n_estimators={n_estimators}, max_depth={max_depth}, accuracy={accuracy:.4f}")
        print(f"  model URI: {model_info.model_uri}")
        # model_info.model_uri = 'models:/m-<id>': direct reference to this run's logged model
        return run.info.run_id, accuracy, model_info.model_uri

In [34]:
print("Training and Logging Model 1...")
run_id_1, acc_1, uri_1 = train_and_log_model(n_estimators=20, max_depth=5, run_name="random_forest_model_run_1")

Training and Logging Model 1...
Logged run 'random_forest_model_run_1' (id=ed1765be85e84266806d1820b962e2c6) n_estimators=20, max_depth=5, accuracy=0.9389
  model URI: models:/m-00519f222daa485a89de6037b09e6c80
🏃 View run random_forest_model_run_1 at: https://dagshub.com/Phaztox/dvc-lab_efrei.mlflow/#/experiments/0/runs/ed1765be85e84266806d1820b962e2c6
🧪 View experiment at: https://dagshub.com/Phaztox/dvc-lab_efrei.mlflow/#/experiments/0


In [35]:
print("Training and Logging Model 2...")
run_id_2, acc_2, uri_2 = train_and_log_model(n_estimators=100, max_depth=10, run_name="random_forest_model_run_V1.0.2")

Training and Logging Model 2...
Logged run 'random_forest_model_run_V1.0.2' (id=8f811b10b78049b8b0d0dffebc611917) n_estimators=100, max_depth=10, accuracy=0.9722
  model URI: models:/m-2839301e774a4d768f218d09009aad49
🏃 View run random_forest_model_run_V1.0.2 at: https://dagshub.com/Phaztox/dvc-lab_efrei.mlflow/#/experiments/0/runs/8f811b10b78049b8b0d0dffebc611917
🧪 View experiment at: https://dagshub.com/Phaztox/dvc-lab_efrei.mlflow/#/experiments/0


## 3. Register the best model in the MLflow Model Registry

Each call to `register_model` with the same name creates a new **version**. MLflow 3 uses **aliases** (e.g. `champion`) instead of the old Staging/Production stages.

> We register with the logged-model URI (`models:/m-...`) returned by `log_model`, not with `runs:/<run_id>/<name>`.
> With a `runs:/` URI, MLflow 3 has to search the experiment for the matching logged model, and that search
> fails on the DagsHub server ("Unable to find a logged_model ..."). The direct URI works locally and on DagsHub.

In [36]:
REGISTERED_MODEL = "RandomForest_Digits"

best_run_id, best_uri = (run_id_1, uri_1) if acc_1 >= acc_2 else (run_id_2, uri_2)
print("Best run:", best_run_id, "| model:", best_uri)

mv = mlflow.register_model(best_uri, REGISTERED_MODEL)
print(f"Registered '{mv.name}' version {mv.version}")

client = MlflowClient()
client.set_registered_model_alias(REGISTERED_MODEL, "champion", mv.version)
print(f"Alias 'champion' -> version {mv.version}")

Best run: 8f811b10b78049b8b0d0dffebc611917 | model: models:/m-2839301e774a4d768f218d09009aad49


Registered model 'RandomForest_Digits' already exists. Creating a new version of this model...
2026/11/18 17:25:56 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: RandomForest_Digits, version 1
Created version '1' of model 'RandomForest_Digits'.


Registered 'RandomForest_Digits' version 1
Alias 'champion' -> version 1


## 4. Load models back from MLflow

In [37]:
# a) From the registry, by alias (what a production service would use)
champion = mlflow.pyfunc.load_model(f"models:/{REGISTERED_MODEL}@champion")
print("champion predictions:", champion.predict(X_test[:10]))
print("true labels:         ", y_test[:10])

# b) From the registry, by version number
by_version = mlflow.pyfunc.load_model(f"models:/{REGISTERED_MODEL}/{mv.version}")

# c) Directly from the best run's logged model
from_run = mlflow.sklearn.load_model(best_uri)
print("accuracy of reloaded model:", accuracy_score(y_test, from_run.predict(X_test)))

champion predictions: [6 9 3 7 2 1 5 2 5 2]
true labels:          [6 9 3 7 2 1 5 2 5 2]
accuracy of reloaded model: 0.9722222222222222


## 5. View the results / serve the model

In [38]:
print("Tracking server:", mlflow.get_tracking_uri())
print("Serve the model (step 5):")
print(f"  mlflow models serve -m 'models:/{REGISTERED_MODEL}@champion' -p 5003")
print(f"  mlflow models serve -m '{best_uri}' -p 5003")

Tracking server: https://dagshub.com/Phaztox/dvc-lab_efrei.mlflow
Serve the model (step 5):
  mlflow models serve -m 'models:/RandomForest_Digits@champion' -p 5003
  mlflow models serve -m 'models:/m-2839301e774a4d768f218d09009aad49' -p 5003
